# Experiment Folder Creator
This file is for loading in a base config from ./base_configs, modifying a few of the fields, and then creating the experiment folder

## Specify Base Config

In [33]:
## Load in base config
from pathlib import Path
import yaml
import copy
import os

# BASE_CONFIG_PATH = Path("base_configs/farming_basic/fb.yaml")
BASE_CONFIG_PATH = Path("base_configs/abstract_bandit/ab.yaml")
# Load yaml file in as dictionary
with open(BASE_CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

# Quick sanity display (in a notebook this will print the dict)
# base_config


## Helper Update Functions

In [11]:
def deep_update(original, update):
    """Deep update original dict with values from the update dict."""
    for key, value in update.items():
        if isinstance(value, dict):
            original[key] = deep_update(original.get(key, {}), value)
        else:
            original[key] = value
    return original

def save_config(config, directory, experiment_name):
    full_path = os.path.join(directory, experiment_name)
    os.makedirs(full_path, exist_ok=True)
    with open(os.path.join(full_path, "config.yaml"), 'w') as f:
        yaml.dump(config, f)

In [12]:
def generate_experiment_name(params, param_grid):
    # Convert each parameter to a string of the form "paramName_paramValue"
    # and join them all with underscores
    # Only keep the fields which have more than one value in param_grid
    params_to_keep = {}
    for param_key in params.keys():
        if len(param_grid[param_key]) > 1: # We assume that the key is in param grid
            params_to_keep[param_key] = params[param_key]

    return "_".join([f"{param}_{value}" for param, value in params_to_keep.items()])

In [13]:
# This config isn't run, but rather just points to the subexperiments - TODO: Decide if this is still relevant
def save_base_config(base_config, experiment_set):
    # Update the master_path and dir keys
    base_config['paths']['eval_results_master_path'] = f"experiments/{experiment_set}/eval_results.csv"
    base_config['paths']['experiment_dir'] = f"experiments/{experiment_set}"
    
    # Write the updated base config to the experiment set directory
    with open(f"experiments/{experiment_set}/config.yaml", 'w') as f:
        yaml.dump(base_config, f)



## Shuffle tools

In [14]:
# Here we add order shuffling for actions
import math, random, itertools

# Lehman unranking for huge spaces - from chatgpt, haven't checked validity
def unrank_permutation(elems, rank):
            elems = list(elems)
            n = len(elems)
            out = []
            r = rank
            for i in range(n-1, -1, -1):
                fact = math.factorial(i)
                idx = r // fact
                r %= fact
                out.append(elems.pop(idx))
            return tuple(out)

def sample_k_orderings(keys, k, *, seed=None):
    keys = list(keys)
    n = len(keys)
    total = math.factorial(n)
    if k > total:
        raise ValueError(f"k ({k}) > number of possible orderings ({total})")
    if seed is not None:
        random.seed(seed)
    # If total is small-ish, enumerate and sample (simple & reproducible)
    if total <= 1000000:
        perms = list(itertools.permutations(keys))
        return random.sample(perms, k)
    else:
        ranks = random.sample(range(total), k)
        return [unrank_permutation(keys, r) for r in ranks]

## Update scale
This function takes a config dict and the path to a scale yaml file and updates the config dict to have those scales for its actions

In [15]:
def update_actions_with_scale(config_dict, scale_path):
    """
    Updates the 'actions' in config_dict with the mean and std_dev values
    from the scale yaml file for matching action IDs.

    Args:
        config_dict (dict): The loaded dictionary of the config file.
        scale_path (str): The path to the scale yaml file.

    Returns:
        dict: The updated config_dict.
    """
    # Load the scale yaml file
    with open(scale_path, 'r') as file:
        scale_data = yaml.safe_load(file)

    # Create a mapping of action IDs to their mean and std_dev in scale
    scale_actions = {action['id']: action for action in scale_data['actions']}

    # Update the subtasks in config_dict
    for subtask in config_dict.get("subtasks", []):
        if 'params' in subtask and 'actions' in subtask['params']:
            for action in subtask['params']['actions']:
                action_id = action['id']
                if action_id in scale_actions:
                    # Update mean and std_dev for matching action IDs
                    action.update({
                        'mean': scale_actions[action_id]['mean'],
                        'std_dev': scale_actions[action_id]['std_dev']
                    })

    return config_dict

## Update Arm Names
This function takes a config dict and the path to an arm name yaml file and updates the config dict to have those scales for its actions

In [16]:
def update_actions_with_arm_name(config_dict, arm_name_path):
    """
    Updates the 'actions' in config_dict with the name
    from the arm_name yaml file for matching action IDs.

    Args:
        config_dict (dict): The loaded dictionary of the config file.
        arm_name_path (str): The path to the arm_name yaml file.

    Returns:
        dict: The updated config_dict.
    """
    # Load the high_arm_name.yaml file
    with open(arm_name_path, 'r') as file:
        arm_name_data = yaml.safe_load(file)

    # Create a mapping of action IDs to their mean and std_dev in high_arm_name
    arm_name_actions = {action['id']: action for action in arm_name_data['actions']}

    # Update the subtasks in fb_basic_dict
    for action in config_dict['actions']:
        action_id = action['id']
        if action_id in arm_name_actions:
            # Update name for matching action IDs
            action.update({
                'names_path': arm_name_actions[action_id]['names_path'],
            })

    return config_dict

## Value mapping table

In [17]:
# Define the mapping from parameter values to specific config changes
value_mapping = {
        'time_horizon': {
            1: {'experiment': {'time_horizon': 1}},
            5: {'experiment': {'time_horizon': 5}},
            8: {'experiment': {'time_horizon': 8}},
            10: {'experiment': {'time_horizon': 10}},
            20: {'experiment': {'time_horizon': 20}}
        },
        'agent': {
            'MonoLLM': {'agent': {'type': 'mono_llm'}},
        },
        'model_name': {
            'Llama3-8B': {'agent': {'model_name': "meta-llama/Meta-Llama-3.1-8B-Instruct"}},
            'Llama2-13B': {'agent': {'model_name': "meta-llama/Llama-2-13b-chat-hf"}},
            # Post-trained Qwen3 - these are the main ones for experiments
            'Qwen3-4B': {'agent': {'model_name': "Qwen/Qwen3-4B"}},
            'Qwen3-8B': {'agent': {'model_name': "Qwen/Qwen3-8B"}},
            'Qwen3-14B': {'agent': {'model_name': "Qwen/Qwen3-14B"}},
            'Qwen3-32B': {'agent': {'model_name': "Qwen/Qwen3-32B"}},
            # Base Qwen3
            'Qwen3-4B-Base': {'agent': {'model_name': "Qwen/Qwen3-4B-Base"}},
            'Qwen3-8B-Base': {'agent': {'model_name': "Qwen/Qwen3-8B-Base"}},
            'Qwen3-14B-Base': {'agent': {'model_name': "Qwen/Qwen3-14B-Base"}},
            # Post-trained Qwen2.5
            'Qwen2.5-7B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-7B-Instruct"}},
            'Qwen2.5-14B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-14B-Instruct"}},
            'Qwen2.5-32B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-32B-Instruct"}},
            'Qwen2.5-72B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-72B-Instruct"}},
            # Base Qwen2.5
            'Qwen2.5-7B': {'agent': {'model_name': "Qwen/Qwen2.5-7B"}},
            'Qwen2.5-14B': {'agent': {'model_name': "Qwen/Qwen2.5-14B"}},
            'Qwen2.5-32B': {'agent': {'model_name': "Qwen/Qwen2.5-32B"}},
        },
        'think_budget': {
            'think_budget_100_150': {'agent': {'thinking_budget': 100,
                                                  'max_new_tokens': 150}},
            'think_budget_150_200': {'agent': {'thinking_budget': 150,
                                                  'max_new_tokens': 200}}
        },
        'prompt':{
            'fullhist': {'world': {'prompt_template_path': 'src/templates/budget_fullhist_v1.jinja2'}},
            'actionhist': {'world': {'prompt_template_path': 'src/templates/budget_actionhist_v1.jinja2'}},
            'summhist': {'world': {'prompt_template_path': 'src/templates/budget_summhist_v1.jinja2'}},\
            'pre-budget': {'world': {'prompt_template_path': 'src/templates/v1_w_hist_budget.jinja2'}},
        },
        'replicates': {
            5: {'experiment': {'replicates': 5}},
            10: {'experiment': {'replicates': 10}},
            20: {'experiment': {'replicates': 20}},
        },
        'temperature': {
            0.0: {'agent': {'temperature': 0.0}},
            1.0: {'agent': {'temperature': 1.0}},
        },
        'scalesweep': {
            True: {'experiment': {'scalesweep': True}},
            False: {'experiment': {'scalesweep': False}},
        }
}

## Param grid - THIS IS WHERE YOU UPDATE
Specify the variations which you want in your experiments. If you specify a single value for a variable (e.g. 'noise': [0]), then all generated configs will have that value. If you specify multiple values (e.g. 'noise': [0, 0.25]), then configs will be generated with each of those values. For example, if you specify 'noise': [0, 0.25] and 'selection': ['ucb', 'thompson'], then you will generate 4 config files - one with each pairwise combination.


In [31]:
# Define the parameter grid
param_grid = {
        # 'model_name': ['Qwen3-14B', 'Qwen3-32B'],
        'model_name': ['Llama3-8B', 'Llama2-13B'],
        'time_horizon': [10],
        'think_budget': ['think_budget_150_200'],
        'prompt': ['summhist'],
        'replicates': [10],
        # 'scalesweep': [True],
    }


In [47]:
# Which scale variations to create
scale_configs = [
    # 'base_configs/scales/no_var/high_scale.yaml',
    # 'base_configs/scales/no_var/low_scale.yaml',
    # 'base_configs/scales/no_var/high_neg_scale.yaml',
    # 'base_configs/scales/no_var/low_neg_scale.yaml',

    # 'base_configs/scales/4sd_var/high_scale.yaml',
    # 'base_configs/scales/4sd_var/low_scale.yaml',
    # 'base_configs/scales/4sd_var/high_neg_scale.yaml',
    # 'base_configs/scales/4sd_var/low_neg_scale.yaml',
    
    'base_configs/scales/2sd_var/high_scale.yaml',
    'base_configs/scales/2sd_var/low_scale.yaml',
    'base_configs/scales/2sd_var/high_neg_scale.yaml',
    'base_configs/scales/2sd_var/low_neg_scale.yaml',

    # 'base_configs/scales/scalesweep/no_var/scale0.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale0.1.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale0.8.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale0.9.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale1.0.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale1.1.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale1.2.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale2.0.yaml',
    # 'base_configs/scales/scalesweep/no_var/scale25.yaml',
]

In [40]:
# arm_name_configs = [
#     'base_configs/farming_basic/arm_names/alphanumeric.yaml',
#     'base_configs/farming_basic/arm_names/ordinal_helpful.yaml',
#     'base_configs/farming_basic/arm_names/ordinal_mislead.yaml',
#     'base_configs/farming_basic/arm_names/world_helpful.yaml',
#     'base_configs/farming_basic/arm_names/world_mislead.yaml',
#     'base_configs/farming_basic/arm_names/sent_helpful.yaml',
#     'base_configs/farming_basic/arm_names/sent_mislead.yaml'
# ]

arm_name_configs = [
    'base_configs/abstract_bandit/arm_names/alphanumeric.yaml',
    'base_configs/abstract_bandit/arm_names/ordinal_helpful.yaml',
    'base_configs/abstract_bandit/arm_names/ordinal_mislead.yaml',
    'base_configs/abstract_bandit/arm_names/sent_helpful.yaml',
    'base_configs/abstract_bandit/arm_names/sent_mislead.yaml'
]

In [ ]:
experiment_set = f'20260124_abandit_2sd_var'

## This function will create the folder

Make sure everything is in order before running. Also make sure your experiment folder name is correctly set to avoid overwriting another folder.


In [48]:
from itertools import permutations, product
import math

# Load the base config file
with open(BASE_CONFIG_PATH, 'r') as f:
    base_config = yaml.safe_load(f)

# Generate and save config files for each combination
for idx, param_values in enumerate(product(*param_grid.values())):
    param_values_dict = dict(zip(param_grid.keys(), param_values))
    experiment_name = generate_experiment_name(param_values_dict, param_grid)
    updated_config = yaml.safe_load(yaml.dump(base_config))  # deep copy

    # Apply updates to the config based on the current parameter values
    for param, value in param_values_dict.items():
        if param in value_mapping and value in value_mapping[param]:
            deep_update(updated_config, value_mapping[param][value])
        else:
            print(f"No mapping found for parameter '{param}' with value '{value}'")

    # For each combination of arm name and scale, create configs
    for arm_path in arm_name_configs:
        arm_base = os.path.splitext(os.path.basename(arm_path))[0]
        for scale_path in scale_configs:
            scale_base = os.path.splitext(os.path.basename(scale_path))[0]
            # Start from the updated_config for this param set
            cfg_combo = yaml.safe_load(yaml.dump(updated_config))  # deep copy
            # Apply arm names and scales
            cfg_combo = update_actions_with_arm_name(cfg_combo, arm_path)
            cfg_combo = update_actions_with_scale(cfg_combo, scale_path)

            # Determine the folder/name suffix including arm and scale bases
            combo_suffix = f"{arm_base}_{scale_base}"

            cfg_combo['paths']['log_file'] = os.path.join(f"experiments/{experiment_set}/{experiment_name}", combo_suffix, "output.log")
            save_config(cfg_combo, f"experiments/{experiment_set}/{experiment_name}", combo_suffix)

# Create a copy of param_grid with only the single item lists and do a deep update
new_base_config = yaml.safe_load(yaml.dump(base_config))  # deep copy
for param_key, param_value in param_grid.items():
    if len(param_value) == 1: # If there is only a single value for this param, change the base config to have it
        if param_key in value_mapping and param_value[0] in value_mapping[param_key]:
            deep_update(new_base_config, value_mapping[param_key][param_value[0]])
         

#saves base config for evaluation purposes        
save_base_config(new_base_config, experiment_set)


In [ ]:
len(list(product(*param_grid.values())))

2